In [127]:
import os
os.environ['HF_HOME'] = '/projectnb/vkolagrp/skowshik/.cache/'
import sys
import logging
import argparse
import warnings
warnings.filterwarnings("ignore")
import yaml
import torch
import json
import inspect
import pandas as pd
import torch.distributed as dist
import torchtune
import re

from tqdm import tqdm
from torch.nn.parallel import DistributedDataParallel as DDP
from datetime import datetime, timedelta
from huggingface_hub import login
from lib.model_loader import load_model, trainer_loader
from lib.data_loader import data_loader_
from utils.utils import CustomStream, load_config
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, LlamaVisionForCausalLM, AutoConfig, LlamaForCausalLM, AutoModel
from lib.model_loader_eval import load_model_eval, load_model_eval_vllm, save_merged_model, load_model_with_llama_proj
from utils.utils import load_json, load_csv
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
from safetensors.torch import load_file
from tokenizers import AddedToken
from peft import LoraConfig, TaskType, PeftModel, get_peft_model
from collections import defaultdict

In [128]:
# config = load_config(file_name="../../code/training/config/config_imaging.yml")
config = load_config(file_name="../../code/training/config/config.yml")
sysmsg = config.get("sysmsg")
dataset_path = "../../data/0825/nacc_unique_with_llama_summaries_test.json"
data = load_json(dataset_path)
# data = load_csv(dataset_path)
max_new_tokens = config.get("max_new_tokens")
hf_write_token = config.get("hf_write_token")
n_devices = 1
max_new_tokens = 4096

In [129]:
nacc_np = pd.read_csv('/projectnb/vkolagrp/datasets/NACC/csv/raw/investigator_ftldlbd_merged_neuropath_nacc65.csv')
# nacc = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/new_nacc_unique_type_3.csv")

In [130]:
# for entry in data:
#     if 'project' in entry['patient_LLAMA_SUMMARY']:
#         print(entry['patient_LLAMA_SUMMARY'])

In [164]:
nacc_np[nacc_np['NACCID'] == 'NACC906696'][['CDRGLOB', 'VISITYR', 'NACCUDSD', 'NACCETPR', 'NACCALZP', 'NACCLEWY', 'NPTHAL', 'NACCBRAA', 'NPCERAD', 'VASCPSIF']]

,CDRGLOB,VISITYR,NACCUDSD,NACCETPR,NACCALZP,NACCLEWY,NPTHAL,NACCBRAA,NPCERAD,VASCPSIF
30449,0.0,2006,1,88,8,2,3,1,NaN,-4
30450,0.0,2007,1,88,8,2,3,1,NaN,-4
30451,0.0,2008,1,88,8,2,3,1,NaN,8
30452,0.0,2009,1,88,8,2,3,1,NaN,8
30453,0.0,2010,1,88,8,2,3,1,NaN,8
30454,0.5,2011,2,2,7,2,3,1,NaN,7
30455,1.0,2012,4,2,7,2,3,1,NaN,7
30456,1.0,2014,4,2,2,2,3,1,NaN,7
30457,1.0,2014,4,2,7,2,3,1,NaN,7
30458,2.0,2015,4,2,7,2,3,1,NaN,-4


In [132]:
example_NACC906696 = load_json("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/0825/all_visits/example_NACC906696.json")

In [133]:
def get_mri_or_emb_dict(val):
    mris_found = defaultdict(int)
    if val == 'mri':
        data_path = '/projectnb/vkolagrp/datasets/NACC/MRI/skullstripped'
        save_path = 'nacc_mri_paths.json'
        extension = '.nii'
    else:
        data_path = '/projectnb/vkolagrp/datasets/NACC/MRI/swin_emb'
        save_path = 'nacc_emb_paths.json'
        extension = '.npy'
    j = 6
    def extract_id(name):
        # This regex matches any characters following "fname-" up until the first occurrence of "_mo"
        match = re.search(r'fname-(.*?)_mo', name)
        if match:
            return match.group(1)    # Returns the captured group which is the part of the string you need
        else:
            match = re.search(r'fname-(.*?)_nodate', name)
            if match:
                return match.group(1) 
        return None

    mri_dict = {}
    # for dirpath, dirs, files in tqdm(os.walk('/projectnb/vkolagrp/datasets/NACC/MRI/skullstripped')): 
    for dirpath, dirs, files in tqdm(os.walk(data_path)): 
        if len(files) == 0:
            continue
        for filename in files:
            fname = os.path.join(dirpath, filename)
            path_list = fname.split('/')
            naccid = path_list[j+1]
            # print(path_list)
            zip_name = extract_id(path_list[j+2])
            modality = path_list[j+3]
            acquisition = path_list[j+4]
            mri_name = path_list[j+5]
            if mri_name in mris_found:
                continue
            mris_found[mri_name] += 1
            # print(naccid, zip_name, modality, acquisition, mri_name)
            
            if modality == 'Unknown' or (not mri_name.endswith(extension)):
                continue
            # if naccid not in mri_dict:
            #     mri_dict[naccid] = {}
            
            if zip_name + 'ni' not in mri_dict:
                mri_dict[zip_name + 'ni'] = {}
            
            mri_dict[zip_name + 'ni'][modality] = {"acquisition": acquisition, "mri_name": fname}


    # with open(save_path, "w") as outfile:
    #     json.dump(mri_dict, outfile, indent=4, sort_keys=False)
        
    return mri_dict, mris_found

In [134]:
emb_dict, mris_found = get_mri_or_emb_dict(val='emb')

21263it [00:10, 1965.21it/s]


In [135]:
import copy
modified_data = copy.deepcopy(example_NACC906696)
for i, entry in enumerate(modified_data):
    zip_name = entry['NACCMRFI']
    if isinstance(zip_name, str):
        mod_zip_name = zip_name.replace(".zip", "ni")
        print(mod_zip_name)
        if mod_zip_name in emb_dict:
            
            print("Modifying")
            mri_text = ''
            for seq, info in emb_dict[mod_zip_name].items():
                if "diffusion" in seq.lower():
                    continue
                mri_text += f"{seq} MRI image: {info['mri_name']} "
            
            # print(mri_text)
            modified_data[i]['patient_LLAMA_SUMMARY'] += f"\n\n**MRI imaging**:\n\n{mri_text}"
            # print(modified_data[i]['patient_LLAMA_SUMMARY'])
            # break

In [136]:
# for entry in modified_data:
#     if 'project' in entry['patient_LLAMA_SUMMARY']:
#         print(entry['patient_LLAMA_SUMMARY'])

## Load model

In [137]:
# llm, tokenizer = load_model_eval_vllm(config, n_devices=2, enable_lora=False, torch_dtype="bfloat16", new_model_path=config.get("model_name"))

In [138]:
# model, tokenizer = load_model_eval(config, base_model=config.get("model_name"), lora_path=config.get("hub_model_id"), torch_dtype=torch.bfloat16, vision=True)
model, tokenizer = load_model_eval(config, base_model=config.get("hub_final_model_id"), lora_path=None, torch_dtype=torch.bfloat16, vision=True)#, save_dir='../../ckpt/fine_tuned_v6_end_to_end_with_mri')

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


Loading LlamaVisionForCausalLM


Loading checkpoint shards: 100%|██████████| 4/4 [00:40<00:00, 10.09s/it]


Copying SwinUNeTR model weights


In [139]:
import re
import numpy as np

def add_image_tokens(df):
    pattern = r'(/\S+\.npy)'
    all_matches = []

    # Iterate over each row in the DataFrame
    for entry in data:
        # Find all matches of the pattern in the 'user' column
        matches = re.findall(pattern, entry['patient_LLAMA_SUMMARY'])
        all_matches += matches
    
    all_matches = list(set(all_matches))
    for match in all_matches:
        try:
            x = np.load(match, mmap_mode='r')
        except:
            print(match)
            raise ValueError
    print(len(all_matches))
    return all_matches

all_matches = add_image_tokens(data)
# all_matches.append('/projectnb/vkolagrp/datasets/NACC/MRI/skullstripped/NACC004099/fname-NACC004099_128401136192134176253428319421300140307886_mo-4_dy-27_yr-2011/T1/3D/1.2.840.113619.2.134.1762534283.1942.1300140307.908_stripped.nii')

new_tokens = set()
if all_matches:
    for matched_string in all_matches:
        token = AddedToken(matched_string, lstrip=False, rstrip=False, normalized=False, special=False)
        new_tokens.add(token)
    tokenizer.add_tokens(list(new_tokens))
    model.resize_token_embeddings(len(tokenizer))

942


In [140]:
def process_user_message(user_msg, reserved_tokens, tokenizer):
    # Define the regular expression pattern for words ending with .nii
    pattern = r'(/\S+\.npy)'
    
    # Define the replacement function
    # It adds "start_of image" before the word, quotes around the word, and "{reserved_tokens}end_of_mri" after
    def replace_func(match):
        matched_string = match.group(1)
        
        # Print the matched string
        # print("Matched string:", matched_string)
        return f'<|start_of_mri|>{matched_string}{reserved_tokens}<|end_of_mri|>'
    
    # Use re.sub to replace all occurrences in the input message
    processed_msg = re.sub(pattern, replace_func, user_msg)
    # print(processed_msg)
    
    return processed_msg

## Testing

In [141]:
index = 2
print(modified_data[index]['patient_LLAMA_SUMMARY'])

**Medical History Summary**

**Subject Demographics:**

* Age: 84 years
* Sex: Female
* Race: White
* Primary language: English
* Level of independence: Able to live independently
* Marital status: Widowed
* Years of education: 16
* Subject's sex: Female
* Subject's month of birth: January
* Subject's year of birth: 1924

**Subject Family History:**

* Father: Had cognitive impairment
* First-degree family member: Had cognitive impairment
* Mother: No report of cognitive impairment

**Subject Medications:**

* Total number of medications reported: 3
* Medications used within two weeks of visit: Vitamin E, Ascorbic Acid, Multivitamin with Minerals

**Subject Health History:**

* Brain trauma - chronic deficit: Absent
* Depression episodes more than two years ago: No
* Brain trauma - brief unconsciousness: Absent
* Brain trauma - extended unconsciousness: Absent
* Other neurological condition: Absent
* Active depression in the last two years: No
* History of traumatic brain injury (TBI):

### Question: Provide the cognitive status at UDS visit for a patient presenting with the following information.

In [167]:
index = 0
print(modified_data[index]['NACCID'])
print(modified_data[index]['NACCUDSD'])

reserved_tokens = '<|reserved_mri_token|>' * (config.get('k') - 1)
user_msg = process_user_message(modified_data[index]['patient_LLAMA_SUMMARY'], reserved_tokens, tokenizer)

NACC906696
1


In [168]:
messages = [
    [
        {"role": "system", "content": sysmsg},
        {"role": "user", "content": f"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nProvide the cognitive diagnosis at UDS visit for a patient presenting with the following information.\n\n### Input:\n{user_msg}"}
    ],
]

In [169]:
prompt = tokenizer.apply_chat_template(
    messages[0], 
    # tokenize=False, 
    return_tensors="pt",
    add_generation_prompt=True
).to("cuda")

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(
        prompt,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.1,
    )
response = outputs[0][prompt.shape[-1]:]
print(tokenizer.decode(response, skip_special_tokens=True).replace(" (NINDS/AIREN criteria)", "").replace(" UDS", ""))

Based on the provided information, the cognitive diagnosis at the visit is:

The patient is cognitively normal for their age, with no impairment in memory, attention, language, or executive function. The patient's cognitive status is deemed normal for age, and there is no evidence of cognitive decline or impairment. The patient's scores on the neuropsychological battery, including the Boston Naming Test, Trail Making Test, and WAIS-R Digit Symbol, are all within normal limits. The patient's MMSE score is 29, which is also within normal limits. The patient's cognitive status is consistent with normal cognition, and there is no evidence of mild cognitive impairment (MCI) or dementia. The patient's diagnosis is normal cognition at all visits.


### Question: Is my patient at risk of progression to dementia?

In [170]:
index = 5
print(modified_data[index]['NACCID'])

reserved_tokens = '<|reserved_mri_token|>' * (config.get('k') - 1)
user_msg = process_user_message(modified_data[index]['patient_LLAMA_SUMMARY'], reserved_tokens, tokenizer)

NACC906696


In [171]:
messages = [
    [
        {"role": "system", "content": sysmsg},
        {"role": "user", "content": f"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nProvide the cognitiva status as normal cognition, mild cognitive impairment or dementia for this patient. Are they at risk of progressing to dementia in the next ten years?.\n\n### Input:\n{user_msg}"}
    ],
]

In [172]:
prompt = tokenizer.apply_chat_template(
    messages[0], 
    # tokenize=False, 
    return_tensors="pt",
    add_generation_prompt=True
).to("cuda")

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(
        prompt,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.2,
    )
response = outputs[0][prompt.shape[-1]:]
print(tokenizer.decode(response, skip_special_tokens=True))

Based on the provided information, the cognitiva status of this patient is Mild Cognitive Impairment (MCI). The patient's cognitive status is deemed to be one or two test scores abnormal, with a total MMSE score of 27 out of 30. The patient's cognitive decline began at the age of 86, and the patient's cognitive status is gradually progressive.

The patient's cognitive impairment is primarily due to Lewy body disease (LBD), which is a contributing cause of the patient's cognitive impairment. The patient's cognitive impairment is also influenced by other factors, including depression, which is a non-contributing cause of the patient's cognitive impairment.

The patient is at risk of progressing to dementia in the next ten years, as the patient's cognitive impairment is gradually progressive and the patient's cognitive status is already mildly impaired. However, the patient's cognitive impairment is not yet severe enough to meet the criteria for dementia.

Overall, the patient's cognitiva

Based on the provided information, the cognitiva status of this patient is Mild Cognitive Impairment (MCI). The patient's cognitive status is deemed to be one or two test scores abnormal, with a total MMSE score of 27 out of 30. The patient's cognitive decline began at the age of 86, and the patient's cognitive status is gradually progressive.

The patient's cognitive impairment is primarily due to Lewy body disease (LBD), which is a contributing cause of the patient's cognitive impairment. The patient's cognitive impairment is also influenced by other factors, including depression, which is a non-contributing cause of the patient's cognitive impairment.

The patient is at risk of progressing to dementia in the next ten years, as the patient's cognitive impairment is gradually progressive and the patient's cognitive status is already mildly impaired. However, the patient's cognitive impairment is not yet severe enough to meet the criteria for dementia.

Overall, the patient's cognitiva status is Mild Cognitive Impairment (MCI) due to Lewy body disease (LBD) and depression, with a gradual progressive course and a risk of progressing to dementia in the next ten years.

### Question: Provide the primary etiologic diagnosis for a patient presenting with the following information.

In [148]:
index = 8
print(modified_data[index]['NACCID'])
print(modified_data[index]['NACCUDSD'])
print(modified_data[index]['NACCETPR'])

reserved_tokens = '<|reserved_mri_token|>' * (config.get('k') - 1)
user_msg = process_user_message(modified_data[index]['patient_LLAMA_SUMMARY'], reserved_tokens, tokenizer)

NACC906696
4
2


In [149]:
messages = [
    [
        {"role": "system", "content": sysmsg},
        {"role": "user", "content": f"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nProvide the primary etiologic diagnosis for a patient presenting with the following information.\n\n### Input:\n{user_msg}"}
    ],
]

In [150]:
prompt = tokenizer.apply_chat_template(
    messages[0], 
    # tokenize=False, 
    return_tensors="pt",
    add_generation_prompt=True
).to("cuda")

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(
        prompt,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.2,
    )
response = outputs[0][prompt.shape[-1]:]
print(tokenizer.decode(response, skip_special_tokens=True))

Based on the provided information, the primary etiologic diagnosis for this patient is Lewy body disease (LBD). The patient has a diagnosis of Parkinson's disease, and the clinician's judgment is that the patient's cognitive decline is due to Lewy body disease. The patient's symptoms, including cognitive impairment, depression, and parkinsonian features, are consistent with LBD. The patient's APOE genotype is e3/e3, which is not associated with an increased risk of LBD. The patient's medications, including donepezil, ropinirole, and entacapone, are not likely to be contributing to their cognitive decline. The patient's medical history, including hypertension and hypercholesterolemia, is not likely to be contributing to their cognitive decline. The patient's family history is significant for cognitive impairment, but the clinician's judgment is that this is not likely to be a contributing factor to the patient's cognitive decline. Overall, the primary etiologic diagnosis for this patien

### Question: Provide the differential diagnosis for a patient presenting with the following information.

In [173]:
index = 8
print(modified_data[index]['NACCID'])
print(modified_data[index]['NACCUDSD'])
print(modified_data[index]['NACCETPR'])

reserved_tokens = '<|reserved_mri_token|>' * (config.get('k') - 1)
user_msg = process_user_message(modified_data[index]['patient_LLAMA_SUMMARY'], reserved_tokens, tokenizer)

NACC906696
4
2


In [176]:
messages = [
    [
        {"role": "system", "content": sysmsg},
        {"role": "user", "content": f"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nProvide ifferential diagnosis for a patient presenting with the following information.\n\n### Input:\n{user_msg}"}
    ],
]

In [177]:
prompt = tokenizer.apply_chat_template(
    messages[0], 
    # tokenize=False, 
    return_tensors="pt",
    add_generation_prompt=True
).to("cuda")

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(
        prompt,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.1,
    )
response = outputs[0][prompt.shape[-1]:]
print(tokenizer.decode(response, skip_special_tokens=True))

Based on the provided information, the patient is a 90-year-old female with a diagnosis of Lewy body disease (LBD) and Parkinson's disease. The patient's cognitive status is deemed to be dementia, and the primary contributing cause of cognitive impairment is Lewy body disease.

The patient's medical history is significant for hypertension, hypercholesterolemia, and a history of smoking. The patient is taking several medications, including anxiolytic, sedative, and hypnotic agents, as well as medications for Parkinson's disease and hypertension.

The patient's physical examination is notable for a moderately stooped posture, mild slowing and/or reduction in amplitude of hand movements, and a slight loss of expression, diction, and/or volume of speech. The patient's gait is slow, and they require little or no assistance.

The patient's neuropsychological battery scores are abnormal, with impairments in attention, executive function, and visuospatial function. The patient's language skill

### Question: Provide the contributing factors for the cognitive decline for a patient presenting with the following information.

In [ ]:
index = 10228
print(data[index]['NACCID'])
print(data[index]['NACCUDSD'])
print(data[index]['instruction'])

reserved_tokens = '<|reserved_mri_token|>' * (config.get('k') - 1)
user_msg = process_user_message(data[index]['patient_LLAMA_SUMMARY'], reserved_tokens, tokenizer)

In [ ]:
messages = [
    [
        {"role": "system", "content": sysmsg},
        {"role": "user", "content": f"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\n{data[index]['instruction']} The MRI images are provided between tokens <|start_of_mri|> and <|end_of_mri|>.\n\n### Input:\n{user_msg}"}
    ],
]

In [ ]:
prompt = tokenizer.apply_chat_template(
    messages[0], 
    # tokenize=False, 
    return_tensors="pt",
    add_generation_prompt=True
).to("cuda")

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(
        prompt,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.2,
    )
response = outputs[0][prompt.shape[-1]:]
print(tokenizer.decode(response, skip_special_tokens=True))